# FFmpeg

A refresher on **FFmpeg** — *the universal Swiss-army knife for audio and video*.
It decodes, encodes, transcodes, mux/demuxes, filters, resamples, trims, and streams
just about any media format that exists. In a speech/audio pipeline it's almost always
the first stage: whatever a model wants (16 kHz mono PCM, a specific codec, a trimmed
clip), `ffmpeg` is what gets you there from the messy file a user actually handed you.

**Domain:** Speech & Audio  ·  **from study list**  ·  **runnable:** yes  ·  _CLI driven from Python via `subprocess`; uses FFmpeg's built-in synthetic sources, so nothing is downloaded_

## 1. What & Why

**What it is.** FFmpeg is a free, cross-platform command-line suite for handling
multimedia. The three binaries you touch:

- **`ffmpeg`** — the workhorse: read inputs, run a filtergraph, encode outputs.
- **`ffprobe`** — inspect a file (codecs, streams, duration, bitrate) without decoding it fully.
- **`ffplay`** — a minimal player (rarely scripted).

Under the hood it's also a set of C libraries (`libavcodec`, `libavformat`,
`libavfilter`, `libswresample`, …) that most of the media world links against —
VLC, Chrome, OBS, and Python wrappers like `pydub`, `torchaudio`'s backend, and
`whisper` all reach for FFmpeg.

**The problem it solves.** Audio/video arrives in a combinatorial mess of
*containers* (`.mp4`, `.mkv`, `.wav`, `.webm`) and *codecs* (`aac`, `opus`, `pcm`,
`mp3`, `h264`). Your model or tool wants exactly one shape. FFmpeg converts between
any of them, resamples, remixes channels, trims, concatenates, and applies filters —
all streaming, so it handles files far larger than RAM.

**When to reach for it.** Preprocessing audio for ASR/TTS (the canonical
"resample to 16 kHz mono WAV"), extracting an audio track from video, format
conversion, batch trimming/segmenting, loudness normalization, or quick inspection
of an unknown file. **When not to:** if you need sample-level DSP in Python
(feature extraction, spectrograms) reach for `librosa`/`torchaudio`; FFmpeg is for
the *transport and transform* layer, not numerical analysis.

## 2. Mental Model

**FFmpeg is a pipeline of pipes.** Every invocation is the same dataflow, left to right:

```
  INPUT(S)          DECODE         FILTERGRAPH         ENCODE          OUTPUT(S)
 ┌─────────┐     ┌──────────┐    ┌────────────┐    ┌──────────┐     ┌──────────┐
 │ in.mp4  │────▶│ demux +  │───▶│ -af/-vf/   │───▶│  encode  │────▶│ out.wav  │
 │ (-i)    │     │ decode   │    │ -filter_   │    │ (codec)  │     │ (mux)    │
 └─────────┘     │ to raw   │    │ complex    │    └──────────┘     └──────────┘
                 └──────────┘    └────────────┘
   container       per-stream      raw frames in,      raw frames        container
   of streams      decoders        raw frames out      -> bytes          of streams
```

Two distinctions make everything click:

1. **Container vs. stream.** A *container* (`.mp4`, `.mkv`) is a box holding one or
   more *streams* (a video stream, one or more audio streams, subtitles). FFmpeg
   *demuxes* the box into streams on input and *muxes* streams back into a box on
   output. Choosing the output container and the per-stream codec are independent
   decisions.

2. **Copy vs. re-encode.** `-c copy` passes encoded packets straight through
   (remux only — instant, lossless, but you can't change the codec or apply filters).
   Anything else *decodes → filters → re-encodes* (slower, lossy for lossy codecs,
   but lets you transform).

Read any FFmpeg command as: **global options · `-i input` · per-output options · output**.
Options *before* `-i` configure the input; options *after* all `-i`s configure the
next output. Order matters.

## 3. Key Concepts

- **Stream selection / mapping** — `-map 0:a:0` picks the first audio stream of input 0.
  Without `-map`, FFmpeg auto-picks "best" streams; with it you control exactly what
  flows to the output. `-vn` drops video, `-an` drops audio.
- **Codec (`-c`)** — `-c:a aac`, `-c:v libx264`, `-c:a pcm_s16le` (WAV), or `-c copy`
  to remux. Suffix targets a stream type: `:a` audio, `:v` video.
- **Resampling & channel layout** — `-ar 16000` (sample rate), `-ac 1` (mono).
  Handled by `libswresample`; the ASR staple is `-ar 16000 -ac 1 -c:a pcm_s16le`.
- **Filtergraphs** — `-af` (audio), `-vf` (video), or `-filter_complex` (multi-in/out).
  Filters chain with `,` and run in parallel with `;`. Examples: `volume=0.5`,
  `atrim`, `loudnorm`, `aresample`, `aformat`.
- **Trimming** — `-ss <start>` and `-t <duration>` (or `-to <end>`). Put `-ss` *before*
  `-i` for a fast keyframe seek; *after* `-i` for sample-accurate (slower) trimming.
- **Rate control** — `-b:a 128k` (target bitrate) vs. `-q:a` (VBR quality) vs.
  `-crf` (constant quality for video). Lower CRF = better/bigger.
- **`lavfi` virtual inputs** — `sine`, `anoisesrc`, `testsrc` synthesize media with no
  file. Invaluable for tests and for this notebook (`-f lavfi -i "sine=..."`).
- **`-y` / `-n`** — overwrite output without asking / never overwrite. Scripts want `-y`.

## 4. Setup

FFmpeg is a **native binary**, not a Python package — install it from your OS package
manager, not pip:

```bash
# macOS
brew install ffmpeg
# Debian/Ubuntu
sudo apt-get install -y ffmpeg
# conda (cross-platform, no admin needed)
conda install -c conda-forge ffmpeg
# Windows
winget install Gyan.FFmpeg     # or: choco install ffmpeg
```

There's no official Python API — you shell out to the binary. Thin wrappers exist
(`ffmpeg-python` builds the arg list as a fluent graph; `imageio-ffmpeg` even bundles
a static binary you can pip-install), but the durable skill is the CLI itself, so this
notebook drives it directly with `subprocess` and FFmpeg's **synthetic `lavfi`
sources** — every example runs offline in a temp dir with no downloads.

In [ ]:
import os, json, shutil, subprocess, tempfile
from pathlib import Path

FFMPEG  = shutil.which("ffmpeg")
FFPROBE = shutil.which("ffprobe")
HAVE_FFMPEG = bool(FFMPEG and FFPROBE)

WORK = Path(tempfile.mkdtemp(prefix="ffmpeg-refresher-"))

def run(*args):
    """Run a command, return (stdout) and raise with stderr on failure."""
    p = subprocess.run(args, capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"{args[0]} failed:\n{p.stderr[-1500:]}")
    return p.stdout

def ffmpeg(*args):
    """ffmpeg with -y (overwrite) and -loglevel error (quiet but real errors)."""
    return run(FFMPEG, "-y", "-loglevel", "error", *args)

def probe(path):
    """ffprobe -> dict of format + streams (the scriptable way to inspect media)."""
    out = run(FFPROBE, "-v", "error", "-print_format", "json",
              "-show_format", "-show_streams", str(path))
    return json.loads(out)

if HAVE_FFMPEG:
    ver = run(FFMPEG, "-version").splitlines()[0]
    print(ver)
    print("scratch dir:", WORK)
else:
    print("FFmpeg not found on PATH — install it (see Setup). Cells below will no-op.")


## 5. Worked Examples

All examples drive the real `ffmpeg`/`ffprobe` binaries on synthetic media generated
in `WORK`. Each cell no-ops cleanly if FFmpeg isn't installed, so the notebook always
executes top-to-bottom.

### Example 1 — Synthesize audio with `lavfi`, then inspect it with `ffprobe`

The two fundamental moves: *make* media and *read* its shape. We generate a 3-second
stereo 44.1 kHz sine tone straight from the `sine` virtual source (no input file),
then probe it. `ffprobe -print_format json` is how you inspect media
**programmatically** — codec, sample rate, channels, and duration all come back as a
dict you can branch on.

In [ ]:
if HAVE_FFMPEG:
    tone = WORK / "tone.wav"
    # -f lavfi -i "sine=..." synthesizes audio; no input file needed.
    ffmpeg("-f", "lavfi", "-i", "sine=frequency=440:duration=3",
           "-ac", "2", "-ar", "44100", "-c:a", "pcm_s16le", str(tone))

    info = probe(tone)
    a = next(s for s in info["streams"] if s["codec_type"] == "audio")
    print(f"file        : {tone.name}  ({tone.stat().st_size/1024:.1f} KiB)")
    print(f"container   : {info['format']['format_name']}")
    print(f"codec       : {a['codec_name']}  ({a.get('sample_fmt')})")
    print(f"sample rate : {a['sample_rate']} Hz")
    print(f"channels    : {a['channels']} ({a.get('channel_layout','?')})")
    print(f"duration    : {float(info['format']['duration']):.2f} s")
else:
    print("skipped (no ffmpeg)")


### Example 2 — Transcode + resample: the ASR-prep one-liner

The single most common FFmpeg task in a speech pipeline: take *whatever* you were
given and produce **16 kHz, mono, 16-bit PCM WAV** — the shape Whisper, wav2vec, and
most STT models expect. The same command also re-encodes the container/codec, so it
doubles as format conversion. We convert our 44.1 kHz stereo tone and probe the
result to prove the transform landed. (`libswresample` does the rate/channel
conversion; the codec change happens because the output is `.wav` / `pcm_s16le`.)

In [ ]:
if HAVE_FFMPEG:
    asr = WORK / "tone_16k_mono.wav"
    # The canonical preprocessing command.
    ffmpeg("-i", str(tone),
           "-ar", "16000",       # resample to 16 kHz
           "-ac", "1",           # downmix stereo -> mono
           "-c:a", "pcm_s16le",  # 16-bit signed PCM
           str(asr))

    before = probe(tone)["streams"][0]
    after  = probe(asr)["streams"][0]
    print(f"{'':12}{'BEFORE':>16}{'AFTER':>16}")
    for k in ("sample_rate", "channels", "codec_name", "sample_fmt"):
        print(f"{k:12}{str(before.get(k)):>16}{str(after.get(k)):>16}")
    print(f"\nsize        {tone.stat().st_size/1024:>13.1f}K{asr.stat().st_size/1024:>15.1f}K")

    # Also show lossy compression in passing: same audio as a 64k MP3.
    mp3 = WORK / "tone.mp3"
    ffmpeg("-i", str(tone), "-c:a", "libmp3lame", "-b:a", "64k", str(mp3))
    print(f"mp3 @64k    {mp3.stat().st_size/1024:.1f} KiB "
          f"(vs {tone.stat().st_size/1024:.1f} KiB WAV — lossy codecs shrink a lot)")
else:
    print("skipped (no ffmpeg)")


### Example 3 — Demux & filter: extract the audio track and trim it with a filtergraph

This shows the **container-vs-stream** model in action. We build a tiny synthetic
*video* (a `testsrc` picture + a `sine` tone muxed together), then pull just the audio
out (`-vn`) while applying a small filtergraph: trim to 1 second (`atrim`) and halve
the volume (`volume`). Filters chain with commas and run in declaration order.
Finally we use `ffprobe`'s `astats` filter to read back the actual peak level — proof
the `volume` filter did something.

In [ ]:
if HAVE_FFMPEG:
    # Build a 2s "video": synthetic 320x240 test pattern + a 330 Hz tone, muxed.
    clip = WORK / "clip.mp4"
    ffmpeg("-f", "lavfi", "-i", "testsrc=size=320x240:rate=15:duration=2",
           "-f", "lavfi", "-i", "sine=frequency=330:duration=2",
           "-c:v", "libx264", "-pix_fmt", "yuv420p",
           "-c:a", "aac", "-shortest", str(clip))

    streams = probe(clip)["streams"]
    kinds = ", ".join(f"{s['codec_type']}:{s['codec_name']}" for s in streams)
    print(f"clip.mp4 contains {len(streams)} streams -> {kinds}")

    # Extract audio only (-vn), trim to 1s and lower volume via an audio filtergraph.
    out = WORK / "extracted.wav"
    ffmpeg("-i", str(clip),
           "-vn",                                    # drop the video stream
           "-af", "atrim=0:1,volume=0.5",            # filter chain: trim, then -6 dB
           "-ar", "16000", "-ac", "1", str(out))

    dur = float(probe(out)["format"]["duration"])
    print(f"extracted.wav: {dur:.2f}s mono 16 kHz (video dropped, audio trimmed)")

    # Measure peak amplitude with the astats filter (output goes to stderr).
    p = subprocess.run([FFMPEG, "-i", str(out), "-af", "astats=metadata=1",
                        "-f", "null", "-"], capture_output=True, text=True)
    peaks = [ln.strip() for ln in p.stderr.splitlines() if "Peak level dB" in ln]
    print("astats:", peaks[-1] if peaks else "(no stats)")
else:
    print("skipped (no ffmpeg)")


## 6. Gotchas & Pitfalls

- **`-ss` placement changes seek behavior.** Before `-i` = fast *keyframe* seek (may
  land slightly off); after `-i` = sample/frame-accurate but slower (it decodes from
  the start). For precise audio trims, put `-ss`/`-t` *after* the input.
- **`-c copy` can't apply filters or change codecs.** Stream copy is remux-only. The
  moment you add `-af`/`-vf` or a different `-c:a`, FFmpeg must re-encode — and for
  lossy codecs (mp3, aac, opus) re-encoding is **generationally lossy**. Avoid
  needless decode→encode round-trips.
- **Option order matters.** Options apply to the *next* file. `ffmpeg -i in.wav -ar 16000 out.wav`
  sets the output rate; `ffmpeg -ar 16000 -i in.wav out.wav` (mis)applies it to the
  *input*. When in doubt, options go right before the file they affect.
- **Forgetting `-y` hangs scripts.** Without `-y`, FFmpeg prompts on an existing
  output and blocks waiting for stdin. Always pass `-y` (or `-n`) in automation.
- **MP3 can't hold raw PCM; WAV can't hold AAC.** Container and codec must be
  compatible. Picking `.mp3` forces an MP3 family codec; if you want PCM, use `.wav`
  or `.flac`. FFmpeg usually errors clearly, but the mapping trips people up.
- **Real output is on stderr.** FFmpeg writes progress/info to **stderr**, not stdout
  (stdout is reserved for piped media). Capture stderr for logs; use `-loglevel error`
  to quiet the banner, or `-progress`/`-nostats` for machine-readable status.
- **Default stream auto-selection can surprise you.** With multiple audio streams
  FFmpeg picks "the best" — not necessarily the one you mean. Use `-map` to be explicit.
- **Variable frame rate & weird sample rates** sneak through; if a downstream model is
  picky, always force `-ar`/`-ac` (audio) or `-r`/`-vf fps=` (video) rather than
  trusting the source.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-off vs FFmpeg |
|---|---|---|
| **FFmpeg (CLI)** | Any format conversion, transcode, resample, trim, mux, extract, batch preprocessing | It's a CLI you shell out to; the option grammar is dense and easy to get subtly wrong |
| **`ffmpeg-python`** | Building complex filtergraphs in Python as a fluent DAG | Just an arg-list builder over the same binary — adds a dependency, hides errors behind its wrapper |
| **`pydub`** | Simple in-Python slicing/concat/gain on short clips | Loads audio into memory; *also* calls FFmpeg under the hood; awkward for large files or video |
| **`librosa` / `torchaudio`** | Sample-level DSP: spectrograms, MFCCs, ML feature extraction | They *analyze* samples; they don't transcode containers/codecs — and torchaudio's I/O backend is often FFmpeg anyway |
| **GUI tools (Audacity, HandBrake)** | One-off manual edits, visual inspection | Not scriptable/reproducible; useless in a pipeline or CI |
| **Cloud media APIs (AWS MediaConvert, etc.)** | Massive-scale, managed transcoding farms | Cost, latency, vendor lock-in; overkill for local preprocessing |

**Rule of thumb:** if the task is "change the *shape/format* of media" (codec,
container, rate, channels, duration, track selection), it's an FFmpeg job. If it's
"compute *numbers from* the samples," it's a librosa/torchaudio job — and FFmpeg
usually feeds them.

## 8. Resources

- **Official docs (full option reference)** — `ffmpeg`, `ffprobe`, every flag: https://ffmpeg.org/ffmpeg.html
- **Filters documentation** — the complete `-af`/`-vf`/`-filter_complex` catalog: https://ffmpeg.org/ffmpeg-filters.html
- **FFmpeg Wiki** — task-oriented recipes (seeking, concatenation, encoding guides): https://trac.ffmpeg.org/wiki
- **Encode/AAC & x264 guides** — practical, opinionated quality/bitrate advice: https://trac.ffmpeg.org/wiki/Encode/AAC and https://trac.ffmpeg.org/wiki/Encode/H.264
- **`ffmpeg-python`** — fluent Python wrapper for building filtergraphs: https://github.com/kkroening/ffmpeg-python

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
CONTAINER_CODECS = {}


def ffmpeg_command(source, output, **kwargs):
    ...


def explain(argv):
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE